# Prompted Segmentation for Drywall QA (CLIPSeg fine-tune) - v2
Prompts: `segment crack` and `segment taping area`.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells **in order, top to bottom**.

v2 changes: grouped split by source photo (no leakage from Roboflow augmented copies), full fine-tuning,
per-prompt threshold tuning on the validation set, failure-case figure, results.json.

In [ ]:
!pip -q install transformers roboflow pycocotools

## 0. Config (edit this cell)

In [ ]:
API_KEY = "PASTE_YOUR_NEW_ROBOFLOW_KEY"   # regenerate your key in Roboflow settings; never share it

# Roboflow locations (crack = your fork, taping = original, version 2)
CRACK_WS, CRACK_PROJ, CRACK_VER = "nawaj-saqlain", "cracks-3ii36-os9wk", 1
TAPE_WS,  TAPE_PROJ,  TAPE_VER  = "objectdetect-pu6rn", "drywall-join-detect", 2

SEED = 42
EPOCHS = 20
LR = 3e-5
BATCH = 16                 # if you get CUDA out of memory, change to 8
UNFREEZE_CLIP = True       # True = fine-tune the whole model, False = decoder only (faster)
MAX_CRACK_TRAIN = 2000     # cap crack training images so cracks don't dominate (None = use all)
SPLIT_FRACS = (0.8, 0.1, 0.1)   # train / val / test, split by SOURCE PHOTO

In [ ]:
import os, random, json, time, re
import numpy as np
import torch, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
import torchvision.transforms as T
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from pycocotools import mask as mask_util
import matplotlib.pyplot as plt

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
def seed_worker(wid):
    np.random.seed(SEED + wid); random.seed(SEED + wid)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| seed:", SEED)

## 1. Download datasets (Roboflow)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
d_crack = rf.workspace(CRACK_WS).project(CRACK_PROJ).version(CRACK_VER).download("coco")
d_tape  = rf.workspace(TAPE_WS).project(TAPE_PROJ).version(TAPE_VER).download("coco")
TAPE_ROOT, CRACK_ROOT = d_tape.location, d_crack.location
print(TAPE_ROOT, CRACK_ROOT)
for r in [CRACK_ROOT, TAPE_ROOT]:
    print(r, "->", sorted(os.listdir(r)))

## 2. Dataset: (image, prompt, mask)

In [ ]:
PROMPTS = {
    "crack": ["segment crack", "segment wall crack", "segment the crack", "crack"],
    "tape":  ["segment taping area", "segment joint/tape", "segment drywall seam", "segment drywall joint"],
}
CANON = {"crack": "segment crack", "tape": "segment taping area"}
SIZE = 352
MEAN, STD = [0.4815, 0.4578, 0.4082], [0.2686, 0.2613, 0.2758]

def ann_to_mask(ann, h, w):
    seg = ann.get("segmentation")
    if isinstance(seg, list) and len(seg) > 0:
        polys = [p for p in seg if len(p) >= 6]
        if polys:
            return mask_util.decode(mask_util.merge(mask_util.frPyObjects(polys, h, w)))
    elif isinstance(seg, dict):
        rle = mask_util.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list) else seg
        return mask_util.decode(rle)
    # fallback for box-only labels: filled bounding box
    x, y, bw, bh = [int(round(v)) for v in ann["bbox"]]
    m = np.zeros((h, w), np.uint8); m[max(y, 0):y + bh, max(x, 0):x + bw] = 1
    return m

class SegDS(Dataset):
    def __init__(self, root, split, kind, train):
        self.dir = os.path.join(root, split)
        coco = json.load(open(os.path.join(self.dir, "_annotations.coco.json")))
        self.imgs = coco["images"]
        self.anns = {}
        for a in coco["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)
        self.kind, self.train = kind, train
        self.jitter = T.ColorJitter(0.3, 0.3, 0.3, 0.05)
        self.norm = T.Normalize(MEAN, STD)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        info = self.imgs[i]
        path = os.path.join(self.dir, info["file_name"])
        img = Image.open(path).convert("RGB")
        w, h = img.size
        m = np.zeros((h, w), np.uint8)
        for a in self.anns.get(info["id"], []):
            m = np.maximum(m, ann_to_mask(a, h, w))
        x = img.resize((SIZE, SIZE), Image.BILINEAR)
        y = Image.fromarray(m * 255).resize((SIZE, SIZE), Image.NEAREST)
        if self.train:
            if random.random() < 0.5: x, y = T.functional.hflip(x), T.functional.hflip(y)
            if random.random() < 0.5: x, y = T.functional.vflip(x), T.functional.vflip(y)
            x = self.jitter(x)
            prompt = random.choice(PROMPTS[self.kind])
        else:
            prompt = CANON[self.kind]
        x = self.norm(T.functional.to_tensor(x))
        y = torch.from_numpy((np.array(y) > 127).astype(np.float32))
        return dict(pixel_values=x, mask=y, prompt=prompt, kind=self.kind, path=path,
                    img_id=info["id"], gt_full=None if self.train else m)

processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")

def collate(batch):
    tok = processor.tokenizer([b["prompt"] for b in batch], padding=True, return_tensors="pt")
    return dict(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"],
                pixel_values=torch.stack([b["pixel_values"] for b in batch]),
                mask=torch.stack([b["mask"] for b in batch]), meta=batch)

## 3. Grouped split (by source photo)
Roboflow exports contain several augmented copies of each photo (`name.rf.<hash>.jpg`).
Copies of one photo always go to the same split, so the test set contains only unseen scenes.

In [ ]:
def grouped_split(root, kind, fracs=SPLIT_FRACS, max_train=None):
    folders = [s for s in ["train", "valid", "test"]
               if os.path.exists(os.path.join(root, s, "_annotations.coco.json"))]
    tr = ConcatDataset([SegDS(root, s, kind, True) for s in folders])
    ev = ConcatDataset([SegDS(root, s, kind, False) for s in folders])
    keys = [info["file_name"].split(".rf.")[0] for d in tr.datasets for info in d.imgs]
    uniq = sorted(set(keys))
    np.random.RandomState(SEED).shuffle(uniq)
    n = len(uniq); n_te, n_va = int(fracs[2] * n), int(fracs[1] * n)
    group = {k: ("test" if i < n_te else "val" if i < n_te + n_va else "train") for i, k in enumerate(uniq)}
    idx = {"train": [], "val": [], "test": []}
    for i, k in enumerate(keys): idx[group[k]].append(i)
    if max_train: idx["train"] = idx["train"][:max_train]
    print(f"{kind} source photos: {n} | images train/val/test: {len(idx['train'])} {len(idx['val'])} {len(idx['test'])}")
    return Subset(tr, idx["train"]), Subset(ev, idx["val"]), Subset(ev, idx["test"])

crack_parts = grouped_split(CRACK_ROOT, "crack", max_train=MAX_CRACK_TRAIN)
tape_parts  = grouped_split(TAPE_ROOT, "tape")
parts = [crack_parts, tape_parts]
train_ds = ConcatDataset([p[0] for p in parts])
val_ds   = ConcatDataset([p[1] for p in parts])
test_ds  = ConcatDataset([p[2] for p in parts])
SPLIT_COUNTS = dict(train=len(train_ds), val=len(val_ds), test=len(test_ds),
                    crack=[len(p) for p in crack_parts], tape=[len(p) for p in tape_parts])
print("split counts ->", SPLIT_COUNTS)

g = torch.Generator(); g.manual_seed(SEED)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, collate_fn=collate,
                      generator=g, worker_init_fn=seed_worker, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)

## 4. Model + metrics

In [ ]:
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined").to(DEVICE)
for n, p in model.named_parameters():
    p.requires_grad = UNFREEZE_CLIP or n.startswith("decoder")
N_TRAIN = sum(p.numel() for p in model.parameters() if p.requires_grad)
N_ALL = sum(p.numel() for p in model.parameters())
print("trainable params: %.1fM / %.1fM" % (N_TRAIN / 1e6, N_ALL / 1e6))

def forward(batch):
    out = model(input_ids=batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE),
                pixel_values=batch["pixel_values"].to(DEVICE))
    lg = out.logits
    return lg.unsqueeze(0) if lg.ndim == 2 else lg

def loss_fn(logits, y):
    bce = F.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2)); den = p.sum((1, 2)) + y.sum((1, 2))
    return bce + (1 - ((2 * inter + 1) / (den + 1)).mean())

def iou_dice(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum(); union = (pred | gt).sum()
    if union == 0: return 1.0, 1.0   # both empty = perfect
    return inter / union, 2 * inter / (pred.sum() + gt.sum())

@torch.no_grad()
def quick_val():
    model.eval(); res = {"crack": [], "tape": []}
    for b in val_dl:
        pr = (torch.sigmoid(forward(b)) > 0.5).cpu().numpy()
        for pm, m in zip(pr, b["meta"]):
            res[m["kind"]].append(iou_dice(pm, m["mask"].numpy())[0])
    return {k: float(np.mean(v)) for k, v in res.items()}

## 5. Train

In [ ]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_dl))
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
best, t0 = -1, time.time()
for ep in range(EPOCHS):
    model.train(); run = 0
    for b in train_dl:
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            lg = forward(b)
        loss = loss_fn(lg.float(), b["mask"].to(DEVICE))
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
    v = quick_val(); score = float(np.mean(list(v.values())))
    print(f"ep {ep+1:02d} | loss {run/len(train_dl):.4f} | val mIoU {v} | mean {score:.4f}")
    if score > best:
        best = score; torch.save(model.state_dict(), "best.pt")
TRAIN_TIME = time.time() - t0
print("train time (min): %.1f" % (TRAIN_TIME / 60))
model.load_state_dict(torch.load("best.pt")); model.eval()

## 6. Tune the threshold per prompt (validation set only)

In [ ]:
@torch.no_grad()
def tune_thresholds():
    model.eval(); probs = {"crack": [], "tape": []}; gts = {"crack": [], "tape": []}
    for b in val_dl:
        p = torch.sigmoid(forward(b)).cpu().numpy()
        for pm, m in zip(p, b["meta"]):
            probs[m["kind"]].append(pm); gts[m["kind"]].append(m["mask"].numpy() > 0.5)
    best_t = {}
    for k in probs:
        scores = {round(float(t), 2): np.mean([iou_dice(p > t, g)[0] for p, g in zip(probs[k], gts[k])])
                  for t in np.arange(0.2, 0.75, 0.05)}
        best_t[k] = max(scores, key=scores.get)
        print(k, "best threshold:", best_t[k], "| val mIoU at 0.5: %.4f -> at best: %.4f" % (scores[0.5], scores[best_t[k]]))
    return best_t

THR = tune_thresholds()

## 7. Test: save PNG masks (0/255, original size) + metrics + timing

In [ ]:
OUT = "predictions"; os.makedirs(OUT, exist_ok=True)
res = {"crack": [], "tape": []}; per_img = []; times = []
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate)
with torch.no_grad():
    for i, b in enumerate(test_dl):
        m = b["meta"][0]; h, w = m["gt_full"].shape
        if DEVICE == "cuda": torch.cuda.synchronize()
        t = time.time()
        lg = forward(b)
        up = F.interpolate(lg[None], size=(h, w), mode="bilinear", align_corners=False)[0, 0]
        pred = (torch.sigmoid(up) > THR[m["kind"]]).cpu().numpy()
        if DEVICE == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t)
        stem = os.path.splitext(os.path.basename(m["path"]))[0]
        fname = f"{m['kind']}_{stem}__{m['prompt'].replace(' ', '_').replace('/', '_')}.png"
        Image.fromarray(pred.astype(np.uint8) * 255).save(os.path.join(OUT, fname))
        r = iou_dice(pred, m["gt_full"]); res[m["kind"]].append(r); per_img.append((m["kind"], r[0], i))

rows = []
for k, name in [("crack", "segment crack"), ("tape", "segment taping area")]:
    a = np.array(res[k]); rows.append((name, len(a), a[:, 0].mean(), a[:, 1].mean()))
allr = np.array(res["crack"] + res["tape"])
rows.append(("ALL (per image)", len(allr), allr[:, 0].mean(), allr[:, 1].mean()))
rows.append(("MEAN of 2 prompts", len(allr), (rows[0][2] + rows[1][2]) / 2, (rows[0][3] + rows[1][3]) / 2))
print(f"{'prompt':22s} {'n':>5s} {'mIoU':>8s} {'Dice':>8s}")
for r in rows: print(f"{r[0]:22s} {r[1]:5d} {r[2]:8.4f} {r[3]:8.4f}")

torch.save(model.state_dict(), "size_check.pt")
MODEL_MB = os.path.getsize("size_check.pt") / 1e6
INFER_MS = 1000 * float(np.mean(times))
print("avg inference time/image: %.1f ms" % INFER_MS)
print("model size: %.1f MB | params: %.1fM" % (MODEL_MB, N_ALL / 1e6))
print("train time: %.1f min | seed: %d | thresholds: %s" % (TRAIN_TIME / 60, SEED, THR))

json.dump(dict(seed=SEED, epochs=EPOCHS, lr=LR, batch=BATCH, unfreeze_clip=UNFREEZE_CLIP,
               split_counts=SPLIT_COUNTS, thresholds=THR,
               metrics={r[0]: dict(n=r[1], mIoU=float(r[2]), Dice=float(r[3])) for r in rows},
               train_minutes=TRAIN_TIME / 60, infer_ms=INFER_MS, model_mb=MODEL_MB,
               params_m=N_ALL / 1e6, trainable_params_m=N_TRAIN / 1e6),
          open("results.json", "w"), indent=2)

## 8. Visual examples (orig | GT | pred) and worst cases for failure notes

In [ ]:
def show_items(items, fname):
    fig, ax = plt.subplots(len(items), 3, figsize=(9, 3 * len(items)))
    ax = np.atleast_2d(ax)
    for r, s in enumerate(items):
        b = collate([s])
        with torch.no_grad(): p = (torch.sigmoid(forward(b))[0] > THR[s["kind"]]).cpu().numpy()
        img = Image.open(s["path"]).convert("RGB").resize((SIZE, SIZE))
        for c, (im, t) in enumerate([(img, "original"), (s["mask"].numpy(), "GT"), (p, f"pred: {s['prompt']}")]):
            ax[r, c].imshow(im, cmap="gray" if c else None); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

rng = random.Random(SEED)
ci = [j for j, (k, _, _) in enumerate(per_img) if k == "crack"]
ti = [j for j, (k, _, _) in enumerate(per_img) if k == "tape"]
sel = rng.sample(ci, 2) + rng.sample(ti, 2)
show_items([test_ds[per_img[j][2]] for j in sel], "examples.png")

In [ ]:
# Worst cases (lowest IoU) for the failure notes: 2 per prompt
worst = sorted([p for p in per_img if p[0] == "crack"], key=lambda x: x[1])[:2] + \
        sorted([p for p in per_img if p[0] == "tape"], key=lambda x: x[1])[:2]
show_items([test_ds[p[2]] for p in worst], "failures.png")

## 9. Download everything

In [ ]:
!zip -qr predictions.zip predictions
from google.colab import files
for f in ["predictions.zip", "examples.png", "failures.png", "results.json"]:
    files.download(f)
# best.pt (about 600 MB) is in the Files panel on the left if you want to download it as well.